# Hierarchischer Ansatz: Weiches Routing

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Prüfen, ob weiches Routing über P(Kultur) · P(Stadium | Kultur) die Fehlerfortpflanzung der harten Kulturentscheidung senkt (#88).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.pipeline import Pipeline

from awp2.config import CROP_COL, HARD_ROUTING, MODEL_DOCS_FIGURES_DIR, SOFT_ROUTING
from awp2.data import band_columns, load_folds, load_split, load_train
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_hierarchical_svm
from awp2.plots import plot_tuning_candidates, save_doc_figure
from awp2.preprocessing import PreprocessingConfig, build_preprocessor

In [ ]:
train = load_train()
bands = band_columns(train)
train.shape

## Hypothese

In `hierarchical_svm` (#87) kosten Kulturfehler rund 0.095 BAcc kombiniert (Oracle-Kultur 0.939
statt 0.844). Beim harten Routing entscheidet allein die Kultur-SVM, welches Stadium-Modell
antwortet. Beim weichen Routing gewinnt das Paar mit dem höchsten P(Kultur) · P(Stadium | Kultur) –
ein sicheres Stadium-Modell kann eine unsichere Kulturentscheidung überstimmen.

Gegenhypothese: Jedes Stadium-Modell kennt nur Spektren seiner Kultur; für fremde Spektren sind
seine Wahrscheinlichkeiten bedeutungslos, aber oft trotzdem hoch. Kulturen mit wenigen Stadien
(Reis: 2) haben außerdem von vornherein höhere P(Stadium | Kultur).

Vorgehen: Dasselbe Modell (SVM mit Wahrscheinlichkeiten, `predict` unverändert) mit `routing` als
Hyperparameter; **nur die CV entscheidet**. Hinweis: Ein Smoke-Test vor diesem Notebook hat
weiches Routing mit den Parametern aus #87 einmal auf der Validierung ausgeführt (0.823); das
fließt nicht in die Entscheidung ein.

In [ ]:
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
param_grid = {
    "routing": [HARD_ROUTING, SOFT_ROUTING],
    "crop_estimator__C": [10.0, 30.0, 100.0],
    "crop_estimator__gamma": [0.003, 0.01, 0.03],
    "stage_estimator__C": [30.0, 100.0],
    "stage_estimator__gamma": [0.003],
}
TOP_CANDIDATES = 15

## CV-Suche: hart gegen weich

In [ ]:
tuned = tune(
    make_hierarchical_svm(SOFT_ROUTING),
    TuneConfig(
        name="hierarchical_soft_search",
        approach="hierarchical",
        description="RBF-SVM → RBF-SVM mit Wahrscheinlichkeiten; routing hart gegen weich bei "
        "gleichem Modell, Kultur-C/gamma und Stadium-C um das Optimum aus #87.",
        preprocessing=preprocessing,
        param_grid=param_grid,
    ),
)
tuned.best_params, tuned.best_score

In [ ]:
candidates = tuned.candidates.assign(
    routing=[params["routing"] for params in tuned.candidates["params"]]
)
best_per_routing = candidates.loc[candidates.groupby("routing")["mean"].idxmax()]
best_per_routing[["routing", "params", "mean", "std", "rank"]]

In [ ]:
tuning_ax = plot_tuning_candidates(
    tuned.candidates.nsmallest(TOP_CANDIDATES, "rank"), color_by="routing"
)
tuning_ax.set_title(f"Hart vs. weich: beste {TOP_CANDIDATES} CV-Kandidaten")
save_doc_figure(tuning_ax.figure, "hierarchical_soft_tuning", MODEL_DOCS_FIGURES_DIR)

## Diagnose auf den Trainings-Folds

Warum schneidet die eine Variante schlechter ab? Out-of-fold-Vorhersagen der jeweils besten
Einstellung je Routing – nur Trainingsdaten, keine Validierung.

In [ ]:
split = load_split()


def out_of_fold_predictions(params: dict) -> pd.DataFrame:
    predictions = []
    for fold in load_folds():
        pipeline = Pipeline(
            [
                ("preprocess", build_preprocessor(preprocessing)),
                ("model", make_hierarchical_svm(SOFT_ROUTING).set_params(**params)),
            ]
        )
        pipeline.fit(split.X_train.iloc[fold.train], split.y_train.iloc[fold.train])
        predictions.append(pipeline.predict(split.X_train.iloc[fold.val]))
    return pd.concat(predictions).loc[split.y_train.index]


oof = {
    row.routing: out_of_fold_predictions(row.params) for row in best_per_routing.itertuples()
}

In [ ]:
true_crop = split.y_train[CROP_COL]
crop_counts = pd.DataFrame(
    {"wahr": true_crop.value_counts()}
    | {f"vorhergesagt ({routing})": pred[CROP_COL].value_counts() for routing, pred in oof.items()}
)
crop_recall = pd.DataFrame(
    {
        routing: (pred[CROP_COL] == true_crop).groupby(true_crop).mean()
        for routing, pred in oof.items()
    }
)
changed = oof[HARD_ROUTING][CROP_COL] != oof[SOFT_ROUTING][CROP_COL]
switches = (
    pd.DataFrame(
        {
            "hart": oof[HARD_ROUTING].loc[changed, CROP_COL],
            "weich": oof[SOFT_ROUTING].loc[changed, CROP_COL],
            "wahr": true_crop[changed],
        }
    )
    .value_counts()
    .head(10)
)
crop_counts, crop_recall.round(3), changed.mean().round(3), switches

**Befund:** Weiches Routing ändert nur 2,3 % der Kulturentscheidungen – und verschiebt sie zur
häufigsten Kultur Mais (Recall Mais 0.916 → 0.935, Baumwolle 0.918 → 0.887). Reis wird *nicht*
bevorzugt; die Gegenhypothese „wenige Stadien“ trifft nicht zu. Plausible Ursache: Die
Platt-Wahrscheinlichkeiten werden ohne Klassengewichte kalibriert und bringen die Klassen-Priors
zurück, die `class_weight="balanced"` aus der Entscheidung herausgenommen hat. Das argmax des
Produkts optimiert damit eher die Accuracy als die Balanced Accuracy.

## Einmalige Bewertung

Nur wenn weiches Routing in der CV gewinnt, wird es einmal mit `run()` bewertet. Gewinnt hartes
Routing, ist der beste Kandidat identisch mit `hierarchical_svm` aus #87 (gleiche Parameter,
`predict` der SVM unabhängig von den Wahrscheinlichkeiten) – ein zweiter Validierungslauf
entfällt.

In [ ]:
if tuned.best_params["routing"] == SOFT_ROUTING:
    result = run(
        tuned.best_model,
        RunConfig(
            name="hierarchical_soft",
            approach="hierarchical",
            description="Bestes weiches Routing aus hierarchical_soft_search, einmal bewertet.",
            preprocessing=preprocessing,
            tuning_run=tuned.run_id,
        ),
    )
    summary = result.metrics.model_dump()
else:
    summary = {"Entscheidung": "hart bleibt; Referenz hierarchical_svm (#87)"}
summary, tuned.run_id